In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Load dataset (requires pyarrow for parquet)
df = pd.read_parquet("../data/yellow_tripdata_2025-01.parquet")
print('shape:', df.shape)
print('columns:', df.columns.tolist())

(3475226, 20) ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


## Initial exploratory analysis

The following cells run descriptive statistics, data quality checks, and list candidate features for predicting hourly pickup demand by zone.

In [ ]:
# 1) Descriptive analysis: overview and basic stats
# Quick peek
display(df.head())

# Basic shape and dtypes
print("Shape:", df.shape)
print("Dtypes:")
print(df.dtypes)

# Missing values by column
missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values (top 20):")
print(missing.head(20))

# Summary statistics for numeric columns
print("Numeric summary:")
display(df.select_dtypes(include=["number"]).describe().T)

# Top pickup zones (if present)
zone_cols = [c for c in df.columns if 'pu' in c.lower() or 'pickup' in c.lower() or 'pulocation' in c.lower()]
print("Candidate pickup zone columns:", zone_cols)
for c in zone_cols:
    print(c, 'unique=', df[c].nunique() if c in df else 'n/a')

# Quick hist of trip distance and passenger_count (if present)
for c in ['trip_distance','passenger_count','fare_amount','total_amount']:
    if c in df.columns:
        print('---', c)
        display(df[c].describe())
        sns.histplot(df[c].dropna(), bins=50)
        plt.title(c)
        plt.show()

In [ ]:
# 2) Data quality checks and detections
issues = {}

# Duplicates
issues['n_duplicates'] = int(df.duplicated().sum())

# Detect datetime-like columns and coerce
dt_candidates = [c for c in df.columns if 'pickup' in c.lower() or 'dropoff' in c.lower() or 'datetime' in c.lower() or 'date' in c.lower()]
issues['datetime_candidates'] = dt_candidates

for c in dt_candidates:
    coerced = pd.to_datetime(df[c], errors='coerce')
    n_coerce = coerced.isna().sum()
    issues[f'na_after_parse_{c}'] = int(n_coerce)
    # attach parsed series for further checks
    df[c+'_parsed'] = coerced

# Range/outlier checks for numeric columns
numeric = df.select_dtypes(include=["number"]).columns.tolist()
issues['numeric_columns'] = numeric[:20]

# Check for non-positive trip_distance or passenger_count
if 'trip_distance' in df.columns:
    issues['trip_distance_nonpos'] = int((df['trip_distance'] <= 0).sum())
if 'passenger_count' in df.columns:
    issues['passenger_count_nonpos'] = int((df['passenger_count'] <= 0).sum())

# Negative monetary values
for money in ['fare_amount','total_amount','tip_amount']:
    if money in df.columns:
        issues[f'neg_{money}'] = int((df[money] < 0).sum())

# Timestamp order check (pickup before dropoff) if both parsed
pick = next((c for c in df.columns if 'pickup' in c.lower() and c.endswith('_parsed')) , None)
drop = next((c for c in df.columns if 'dropoff' in c.lower() and c.endswith('_parsed')) , None)
if pick and drop:
    issues['pickup_after_dropoff'] = int((df[pick] > df[drop]).sum())

# Show issues summary
from pprint import pprint
pprint(issues)

# Display examples of problematic rows for a few checks
if issues.get('n_duplicates',0) > 0:
    display(df[df.duplicated(keep=False)].head())
if 'trip_distance_nonpos' in issues and issues['trip_distance_nonpos']>0:
    display(df[df['trip_distance']<=0].head())
if 'passenger_count_nonpos' in issues and issues['passenger_count_nonpos']>0:
    display(df[df['passenger_count']<=0].head())

## 2) Data quality issues — what to look for and how to detect

- **Missing values:** detect with `df.isnull().sum()`; critical for datetime or zone columns.
- **Duplicates:** detect with `df.duplicated()`; remove exact duplicates or aggregate if intentional.
- **Invalid timestamps:** coerce with `pd.to_datetime(..., errors='coerce')` and check coerced NA counts and order (pickup vs dropoff).
- **Outliers / impossible values:** check numeric ranges (e.g., `trip_distance <= 0`, negative fares, extremely large distances or durations).
- **Schema inconsistencies:** unexpected zone IDs, mixed data types; use `df.dtypes` and `df['zone_col'].unique()` to inspect.
- **Temporal gaps / timezone issues:** plot time series of counts to reveal gaps or shifts.

Mitigation approaches: imputation, filtering (remove erroneous rows), feature engineering (cap/transform outliers), and careful timezone-aware parsing.

## 3) Which columns are useful for predicting hourly demand by pickup zone?

Useful columns and why:

- **Pickup datetime (e.g., `pickup_datetime`):** primary signal to derive hour-of-day, day-of-week, holidays, and seasonal patterns.
- **Pickup zone / `PULocationID` / pickup_zone:** the spatial unit for demand aggregation and the modeling target (pickups per zone).
- **Passenger count:** can reflect group sizes and may correlate with demand intensity.
- **Trip distance / duration:** helps distinguish short local trips vs long trips that remove vehicles from service.
- **Fare / total amount:** proxy for trip value and may correlate with demand patterns (e.g., surge pricing).
- **Vendor / taxi type:** different fleets may serve different demand patterns.
- **Store_and_fwd_flag / rate_code:** indicators of special trip handling or service type.

Additional external features to join (recommended): weather, public events, transit outages, and nearby POI/land-use.

Derived features to build:
- Lagged counts per zone (1h, 24h), rolling means, and differences.
- Time-based features: hour, weekday, is_weekend, is_holiday.
- Spatial context: neighboring-zone demand, distance to central business districts, and density metrics.